# LeetCode 425 — Word Squares (Hard): nhập môn Backtracking + bảng tiền tố (Hash Map / Trie) bằng Python

> **Lưu ý:** đây là bài **Premium** (bị khóa trên LeetCode, cần tài khoản trả phí mới mở đề và nộp được). Notebook chép lại đề, ví dụ, constraints và signature theo bản LeetCode, đồng thời tự kiểm thử mọi thứ ngay trong notebook, nên bạn vẫn học trọn vẹn mà không cần tài khoản Premium.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def wordSquares(self, words: List[str]) -> List[List[str]]:

```

Bài này nghe tên thì lạ, nhưng thực chất là một bài **backtracking** (quay lui) rất "sạch": ta xếp từng từ vào từng hàng của một hình vuông chữ, và ở mỗi hàng, các hàng trước đã **ép buộc** một phần của hàng mới. Cái khó (Hard) là làm sao tìm **nhanh** các từ thỏa phần bị ép buộc đó.

---

## Sau notebook này, bạn sẽ nắm được

1. **Word square** là gì: hàng `k` đọc giống hệt cột `k`, và vì sao điều đó tương đương với "hình vuông chữ **đối xứng** qua đường chéo chính".
2. Nền tảng Python: lấy **cột** của một list chuỗi, `zip(*rows)`, `itertools.product` (chọn **có lặp lại**) khác `permutations` thế nào.
3. **Cách 1 — Brute force**: thử mọi bộ `L` từ rồi kiểm tra; thấy tận mắt vì sao `N^L` là quá lớn.
4. **Insight then chốt**: khi đã đặt các hàng `0..r-1`, hàng `r` **bắt buộc** phải bắt đầu bằng một **tiền tố** (prefix) xác định — đó chính là các ký tự ở cột `r` của những hàng trước.
5. **Backtracking từ căn bản** (chọn → đi tiếp → bỏ chọn) và **Cách 2**: quét mọi từ để tìm từ có tiền tố đúng.
6. **Cách 3 (cách chính)**: `dict` tiền tố → danh sách từ (`defaultdict(list)`), tra cứu tiền tố trung bình O(1) thay vì quét O(N).
7. **Cách 4**: **Trie** (cây tiền tố) lưu danh sách từ tại mỗi node — so sánh bộ nhớ với Hash Map.
8. Trace từng bước trên `["area","lead","wall","lady","ball"]`, code nộp LeetCode, test đối chiếu chéo, vì sao đúng, độ phức tạp, lỗi thường gặp, biến thể và **6 bài tập** có đáp án ở cuối.

## 1. Đề bài

Cho một mảng các chuỗi **không trùng nhau** `words`, hãy trả về **tất cả** các **word square** (hình vuông chữ) có thể tạo được từ `words`. **Một từ trong `words` có thể được dùng nhiều lần.** Bạn có thể trả kết quả theo **thứ tự bất kỳ**.

Một dãy chuỗi tạo thành một **word square hợp lệ** nếu **hàng thứ `k` và cột thứ `k` đọc ra cùng một chuỗi**, với `0 <= k < max(numRows, numColumns)`.

Ví dụ: dãy `["ball","area","lead","lady"]` là một word square vì mỗi từ đọc theo hàng ngang và theo cột dọc đều giống nhau.

**Example 1**

```text
Input: words = ["area","lead","wall","lady","ball"]
Output: [["ball","area","lead","lady"],["wall","area","lead","lady"]]
Explanation: Kết quả gồm hai word square. Thứ tự các word square trong kết quả
không quan trọng (chỉ thứ tự các từ BÊN TRONG mỗi word square là quan trọng).
```

**Example 2**

```text
Input: words = ["abat","baba","atan","atal"]
Output: [["baba","abat","baba","atal"],["baba","abat","baba","atan"]]
Explanation: Kết quả gồm hai word square. Thứ tự không quan trọng.
```

**Constraints**

- `1 <= words.length <= 1000`
- `1 <= words[i].length <= 4`
- Tất cả `words[i]` có **cùng độ dài**.
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- Mọi `words[i]` là **khác nhau** (unique).

Quy ước ký hiệu dùng suốt notebook:

- `N = len(words)`: số từ (tối đa 1000);
- `L = len(words[0])`: độ dài mỗi từ (tối đa 4) — cũng chính là **số hàng** của mỗi word square (giải thích ở mục 1.2);
- `square`: list các từ đã đặt, `square[r]` là hàng `r`, `square[r][c]` là ký tự ở hàng `r`, cột `c`.

Vì thứ tự kết quả tùy ý, trong notebook ta luôn so sánh **sau khi sắp xếp** (`sorted`).

In [1]:
from typing import List

words1 = ["area", "lead", "wall", "lady", "ball"]
expected1 = [["ball", "area", "lead", "lady"], ["wall", "area", "lead", "lady"]]

words2 = ["abat", "baba", "atan", "atal"]
expected2 = [["baba", "abat", "baba", "atal"], ["baba", "abat", "baba", "atan"]]

def norm(squares):
    # Chuẩn hóa để so sánh không phụ thuộc thứ tự: list các tuple, đã sắp xếp
    return sorted(tuple(sq) for sq in squares)

print("N =", len(words1), "| L =", len(words1[0]))
print("Từ thứ 4:", words1[4], "| ký tự thứ 1 của nó:", words1[4][1])
print("Hai cách viết cùng một đáp án có bằng nhau sau norm?",
      norm(expected1) == norm(list(reversed(expected1))))

N = 5 | L = 4
Từ thứ 4: ball | ký tự thứ 1 của nó: a
Hai cách viết cùng một đáp án có bằng nhau sau norm? True


### 1.1 Nhìn tận mắt một word square

Xếp 4 từ `ball, area, lead, lady` thành 4 hàng, mỗi ký tự một ô:

```text
      c0 c1 c2 c3
r0    b  a  l  l     <- hàng 0 = "ball"
r1    a  r  e  a     <- hàng 1 = "area"
r2    l  e  a  d     <- hàng 2 = "lead"
r3    l  a  d  y     <- hàng 3 = "lady"
```

Đọc **cột** `c0` từ trên xuống: `b, a, l, l` → `"ball"` = hàng 0. Cột `c1`: `a, r, e, a` → `"area"` = hàng 1. Cột `c2`: `l, e, a, d` → `"lead"`; cột `c3`: `l, a, d, y` → `"lady"`. Bốn cột trùng bốn hàng → hợp lệ.

Hàm `show_square` dưới đây in đúng hình trên và in thêm các chuỗi cột để bạn so sánh.

In [2]:
def show_square(rows):
    L = len(rows[0])
    print("     " + " ".join(f"c{c}" for c in range(L)))
    for r, w in enumerate(rows):
        print(f"r{r}   " + "  ".join(w) + f"     <- hàng {r} = {w!r}")
    cols = ["".join(w[c] for w in rows) for c in range(len(rows[0]))]
    print("Các cột đọc từ trên xuống:", cols)
    print("Hàng == cột ở mọi k?", list(rows) == cols)

show_square(["ball", "area", "lead", "lady"])
print()
show_square(["ball", "area", "read", "lady"])   # đổi 'lead' -> 'read'

     c0 c1 c2 c3
r0   b  a  l  l     <- hàng 0 = 'ball'
r1   a  r  e  a     <- hàng 1 = 'area'
r2   l  e  a  d     <- hàng 2 = 'lead'
r3   l  a  d  y     <- hàng 3 = 'lady'
Các cột đọc từ trên xuống: ['ball', 'area', 'lead', 'lady']
Hàng == cột ở mọi k? True

     c0 c1 c2 c3
r0   b  a  l  l     <- hàng 0 = 'ball'
r1   a  r  e  a     <- hàng 1 = 'area'
r2   r  e  a  d     <- hàng 2 = 'read'
r3   l  a  d  y     <- hàng 3 = 'lady'
Các cột đọc từ trên xuống: ['barl', 'area', 'lead', 'lady']
Hàng == cột ở mọi k? False


### 1.2 Hai hệ quả quan trọng của định nghĩa

**(a) Word square = hình vuông chữ đối xứng qua đường chéo chính.**

"Hàng `k` = cột `k`" nghĩa là: ký tự thứ `j` của hàng `k` bằng ký tự thứ `j` của cột `k`. Mà ký tự thứ `j` của cột `k` chính là `square[j][k]` (hàng `j`, cột `k`). Vậy điều kiện là:

$$\texttt{square[k][j]} = \texttt{square[j][k]} \quad \text{với mọi } j, k.$$

Ví dụ trong `ball/area/lead/lady`: `square[0][2] = 'l'` (hàng 0, cột 2) và `square[2][0] = 'l'` (hàng 2, cột 0) — bằng nhau. Hai ô "soi gương" qua đường chéo `b, r, a, y` luôn giống nhau. Còn các ô **trên** đường chéo (`j == k`) thì tự động thỏa.

**(b) Số hàng phải đúng bằng `L` (độ dài mỗi từ).**

Cột 0 phải đọc ra hàng 0, mà hàng 0 có `L` ký tự → cột 0 phải có `L` ký tự → phải có đúng `L` hàng. Với `L = 4` ta luôn chọn **đúng 4 từ** (có thể lặp lại). Đừng nhầm `L` (độ dài từ) với `N` (số từ trong input)!

Cell dưới liệt kê mọi cặp ô đối xứng `(k, j)` – `(j, k)` với `j > k` của hình vuông hợp lệ và hình vuông sai ở trên.

In [3]:
def mirror_pairs(rows):
    L = len(rows)
    for k in range(L):
        for j in range(k + 1, L):
            a, b = rows[k][j], rows[j][k]
            mark = "✔" if a == b else "✘"
            print(f"  ô({k},{j})={a!r}  vs  ô({j},{k})={b!r}  {mark}")

print("ball/area/lead/lady:")
mirror_pairs(["ball", "area", "lead", "lady"])
print("ball/area/read/lady:")
mirror_pairs(["ball", "area", "read", "lady"])

ball/area/lead/lady:
  ô(0,1)='a'  vs  ô(1,0)='a'  ✔
  ô(0,2)='l'  vs  ô(2,0)='l'  ✔
  ô(0,3)='l'  vs  ô(3,0)='l'  ✔
  ô(1,2)='e'  vs  ô(2,1)='e'  ✔
  ô(1,3)='a'  vs  ô(3,1)='a'  ✔
  ô(2,3)='d'  vs  ô(3,2)='d'  ✔
ball/area/read/lady:
  ô(0,1)='a'  vs  ô(1,0)='a'  ✔
  ô(0,2)='l'  vs  ô(2,0)='r'  ✘
  ô(0,3)='l'  vs  ô(3,0)='l'  ✔
  ô(1,2)='e'  vs  ô(2,1)='e'  ✔
  ô(1,3)='a'  vs  ô(3,1)='a'  ✔
  ô(2,3)='d'  vs  ô(3,2)='d'  ✔


Nhận xét: với `L = 4` chỉ có `4·3/2 = 6` cặp ô cần so sánh (các ô trên đường chéo không cần so). Hình vuông thứ hai sai **đúng 1 cặp**: `ô(0,2) = 'l'` (chữ thứ 2 của `"ball"`) nhưng `ô(2,0) = 'r'` (chữ đầu của `"read"`). Đổi `lead → read` chỉ làm hỏng chữ **đầu** của hàng 2 — mà chữ đầu hàng 2 bắt buộc phải bằng `ball[2] = 'l'`. Năm cặp còn lại vẫn khớp.

Đây là hạt giống của **insight** chính trong bài: **chữ đầu của hàng 2 bị hàng 0 ép buộc**. Ta sẽ quay lại ý này ở mục 5.

---

## 2. Nền tảng 1 — Chuỗi, chỉ số và cách lấy **cột** của một list chuỗi

Trong Python, chuỗi (string) giống một dãy ký tự có đánh số từ 0: `"ball"[0] == 'b'`, `"ball"[3] == 'l'`.

Một list chuỗi `rows = ["ball", "area", ...]` có thể xem như **lưới ký tự**: `rows[r][c]` là hàng `r`, cột `c`. Để lấy **cột `c`**, ta đi qua từng hàng và nhặt ký tự thứ `c`:

```python
col = "".join(w[c] for w in rows)
```

- `w[c] for w in rows` là **generator expression**: lần lượt sinh `rows[0][c], rows[1][c], ...`.
- `"".join(...)` nối các ký tự đó thành một chuỗi (dùng chuỗi rỗng `""` làm "keo dán").

Một cách khác rất Python: `zip(*rows)`. Dấu `*` "trải" list ra thành các đối số riêng: `zip(*["ab", "cd"])` giống `zip("ab", "cd")`, và `zip` ghép các ký tự **cùng vị trí** → `('a','c')`, `('b','d')` — chính là các **cột**. Phép biến hàng thành cột này gọi là **chuyển vị** (transpose).

In [4]:
rows = ["ball", "area", "lead", "lady"]
print("rows[2]    =", rows[2], "  (hàng 2)")
print("rows[2][0] =", rows[2][0], "  (hàng 2, cột 0)")

c = 1
print("Cột 1 nhặt từng hàng:", [w[c] for w in rows])
print("Cột 1 nối lại       :", "".join(w[c] for w in rows))

print("zip(*['ab','cd'])   :", list(zip(*["ab", "cd"])))
print("Mọi cột qua zip     :", ["".join(col) for col in zip(*rows)])

rows[2]    = lead   (hàng 2)
rows[2][0] = l   (hàng 2, cột 0)
Cột 1 nhặt từng hàng: ['a', 'r', 'e', 'a']
Cột 1 nối lại       : area
zip(*['ab','cd'])   : [('a', 'c'), ('b', 'd')]
Mọi cột qua zip     : ['ball', 'area', 'lead', 'lady']


### 2.1 Hàm kiểm tra `is_word_square`

Từ định nghĩa, ta viết hàm kiểm tra một dãy từ có phải word square không:

1. Số hàng phải bằng độ dài mỗi từ (hệ quả (b)).
2. Với mỗi `k`, cột `k` phải bằng hàng `k`.

Hàm này là "trọng tài" — ta dùng nó cho brute force và để kiểm chứng các cách giải sau.

In [5]:
def is_word_square(rows):
    L = len(rows)
    if any(len(w) != L for w in rows):          # số hàng phải = độ dài từ
        return False
    for k in range(L):
        column_k = "".join(w[k] for w in rows)
        if column_k != rows[k]:
            return False
    return True

print(is_word_square(["ball", "area", "lead", "lady"]))   # True
print(is_word_square(["wall", "area", "lead", "lady"]))   # True
print(is_word_square(["ball", "area", "read", "lady"]))   # False
print(is_word_square(["baba", "abat", "baba", "atan"]))   # True (dùng 'baba' 2 lần)
print(is_word_square(["ball", "area", "lead"]))            # False: 3 hàng nhưng từ dài 4
print(is_word_square(["a"]))                                # True: hình vuông 1x1

True
True
False
True
False
True


## 3. Nền tảng 2 — `itertools.product` (chọn **có lặp**) và `permutations` (chọn **không lặp**)

Đề nói: **một từ có thể dùng nhiều lần**. Example 2 dùng `"baba"` ở cả hàng 0 và hàng 2. Vì vậy khi "thử mọi bộ `L` từ", ta cần **tích Descartes** (Cartesian product): mỗi hàng chọn độc lập một từ bất kỳ.

- `product(words, repeat=L)`: mọi bộ `L` từ, **cho phép lặp**, có thứ tự → `N^L` bộ.
- `permutations(words, L)`: mọi bộ `L` từ **khác nhau**, có thứ tự → `N·(N-1)·...·(N-L+1)` bộ.

Với `words = ["x", "y"]` và `L = 2`: `product` cho `xx, xy, yx, yy` (4 bộ) còn `permutations` chỉ cho `xy, yx` (2 bộ) — mất `xx` và `yy`. Nếu dùng nhầm `permutations`, ta sẽ **mất đáp án** của Example 2 (xem lại ở phần Lỗi thường gặp).

In [6]:
from itertools import product, permutations

small = ["x", "y"]
print("product     :", list(product(small, repeat=2)))
print("permutations:", list(permutations(small, 2)))

print("Số bộ product(words1, 4)      =", len(list(product(words1, repeat=4))), "= 5^4")
print("Số bộ permutations(words1, 4) =", len(list(permutations(words1, 4))), "= 5*4*3*2")

product     : [('x', 'x'), ('x', 'y'), ('y', 'x'), ('y', 'y')]
permutations: [('x', 'y'), ('y', 'x')]
Số bộ product(words1, 4)      = 625 = 5^4
Số bộ permutations(words1, 4) = 120 = 5*4*3*2


## 4. Cách 1 — Brute force: thử mọi bộ `L` từ rồi kiểm tra

Ý tưởng tự nhiên nhất mà người mới nghĩ ra:

1. Gọi `L = len(words[0])` — số hàng cần chọn.
2. Duyệt **mọi** bộ `(w0, w1, ..., w_{L-1})` bằng `product(words, repeat=L)`.
3. Bộ nào qua được `is_word_square` thì thêm vào kết quả.

Cách này chắc chắn đúng (thử hết thì không sót), và rất hữu ích làm **đáp án chuẩn** để kiểm tra các cách nhanh hơn trên input nhỏ.

In [7]:
def word_squares_bruteforce(words):
    L = len(words[0])
    result = []
    for combo in product(words, repeat=L):     # combo là tuple L từ
        if is_word_square(combo):
            result.append(list(combo))
    return result

res1 = word_squares_bruteforce(words1)
res2 = word_squares_bruteforce(words2)
print(res1)
print(res2)
assert norm(res1) == norm(expected1)
assert norm(res2) == norm(expected2)
print("Brute force: PASS cả 2 ví dụ")

[['wall', 'area', 'lead', 'lady'], ['ball', 'area', 'lead', 'lady']]
[['baba', 'abat', 'baba', 'atan'], ['baba', 'abat', 'baba', 'atal']]
Brute force: PASS cả 2 ví dụ


### Nút thắt của cách 1

Số bộ phải thử là `N^L`, mỗi bộ kiểm tra mất `O(L^2)` (so `L` cột, mỗi cột `L` ký tự):

| `N` | `L` | Số bộ `N^L` | Ước lượng (≈ 10^7 bộ/giây trong Python) |
|---:|---:|---:|---|
| 5 | 4 | 625 | tức thì |
| 100 | 4 | 10^8 | ~10 giây |
| 1000 | 4 | 10^12 | ~1 ngày |

Với constraints `N <= 1000`, `L <= 4`, brute force **không thể** qua. Vấn đề cốt lõi: nó chọn **cả 4 hàng** rồi mới kiểm tra. Ví dụ hàng 0 là `"area"` thì **không có** từ nào làm được hàng 1 (sẽ thấy ngay dưới đây), nhưng brute force vẫn ngây thơ thử `5^3 = 125` cách chọn 3 hàng còn lại cho mỗi lựa chọn hàng 0 như vậy.

Ta cần phát hiện ngõ cụt **càng sớm càng tốt** — ngay khi vừa đặt một hàng.

## 5. Insight then chốt — hàng `r` bị các hàng trước **ép tiền tố**

Giả sử ta xây hình vuông **từ trên xuống**, đã đặt xong các hàng `0, 1, ..., r-1`. Hàng `r` cần thỏa gì?

Theo tính đối xứng `square[r][j] = square[j][r]`: với mọi `j < r`, **ký tự thứ `j` của hàng `r`** phải bằng `square[j][r]` — ký tự ở **cột `r`** của hàng `j` (đã biết!). Vậy `r` ký tự đầu của hàng `r` đã bị **xác định hoàn toàn**:

```python
prefix = "".join(square[j][r] for j in range(r))   # = cột r của các hàng đã đặt
```

Ta chỉ được chọn hàng `r` là một từ **bắt đầu bằng `prefix`** (prefix = **tiền tố**, phần đầu của chuỗi).

Ví dụ với `words1`:

| Các hàng đã đặt | `r` | Tiền tố bắt buộc của hàng `r` | Lấy từ đâu | Ứng viên |
|---|---:|---|---|---|
| (chưa có) | 0 | `""` | không có ràng buộc | cả 5 từ |
| `ball` | 1 | `"a"` | `ball[1]` | `area` |
| `ball, area` | 2 | `"le"` | `ball[2], area[2]` | `lead` |
| `ball, area, lead` | 3 | `"lad"` | `ball[3], area[3], lead[3]` | `lady` |
| `area` | 1 | `"r"` | `area[1]` | **không có** → ngõ cụt ngay! |
| `lady, area` | 2 | `"de"` | `lady[2], area[2]` | **không có** → ngõ cụt |

Chú ý cách tính: tiền tố của hàng `r` = **cột `r`** của các hàng đã đặt — đúng y công thức lấy cột ở mục 2.

In [8]:
def required_prefix(square):
    r = len(square)                      # hàng sắp đặt
    return "".join(row[r] for row in square)

for partial in [[], ["ball"], ["ball", "area"], ["ball", "area", "lead"], ["area"], ["lady", "area"]]:
    p = required_prefix(partial)
    cands = [w for w in words1 if w.startswith(p)]
    print(f"đã đặt {partial!s:<26} -> hàng {len(partial)} phải bắt đầu bằng {p!r:<6} -> ứng viên {cands}")

đã đặt []                         -> hàng 0 phải bắt đầu bằng ''     -> ứng viên ['area', 'lead', 'wall', 'lady', 'ball']
đã đặt ['ball']                   -> hàng 1 phải bắt đầu bằng 'a'    -> ứng viên ['area']
đã đặt ['ball', 'area']           -> hàng 2 phải bắt đầu bằng 'le'   -> ứng viên ['lead']
đã đặt ['ball', 'area', 'lead']   -> hàng 3 phải bắt đầu bằng 'lad'  -> ứng viên ['lady']
đã đặt ['area']                   -> hàng 1 phải bắt đầu bằng 'r'    -> ứng viên []
đã đặt ['lady', 'area']           -> hàng 2 phải bắt đầu bằng 'de'   -> ứng viên []


### 5.1 Chỉ cần kiểm tra tiền tố là đủ — không cần kiểm tra lại cả hình vuông ở cuối

Câu hỏi tự nhiên: nếu mỗi lần đặt hàng `r` ta chỉ kiểm tra "hàng `r` bắt đầu bằng `prefix`", thì khi đặt đủ `L` hàng, hình vuông có **chắc chắn** hợp lệ không?

**Có.** Khi đặt hàng `r`, phép kiểm tra tiền tố bảo đảm `square[r][j] == square[j][r]` cho **mọi `j < r`** — tức là mọi cặp ô đối xứng có một đầu nằm ở hàng `r` và đầu kia ở hàng **phía trên**. Mỗi cặp ô `(i, j)` với `i > j` được kiểm tra **đúng một lần**: lúc đặt hàng `i` (hàng thấp hơn trong hai hàng). Các ô trên đường chéo tự thỏa. Vậy sau khi đặt đủ `L` hàng, **mọi** cặp đều đã được kiểm tra → hình vuông đối xứng → hợp lệ.

Cell dưới in ra: khi đặt hàng `r`, những cặp ô nào được "chốt".

In [9]:
L = 4
for r in range(L):
    pairs = [f"({r},{j})=({j},{r})" for j in range(r)]
    print(f"Đặt hàng {r}: tiền tố dài {r} -> chốt các cặp {pairs if pairs else '(không có)'}")
total = sum(range(L))
print("Tổng số cặp được chốt:", total, "= L*(L-1)/2 =", L * (L - 1) // 2, "-> đủ mọi cặp ngoài đường chéo")

Đặt hàng 0: tiền tố dài 0 -> chốt các cặp (không có)
Đặt hàng 1: tiền tố dài 1 -> chốt các cặp ['(1,0)=(0,1)']
Đặt hàng 2: tiền tố dài 2 -> chốt các cặp ['(2,0)=(0,2)', '(2,1)=(1,2)']
Đặt hàng 3: tiền tố dài 3 -> chốt các cặp ['(3,0)=(0,3)', '(3,1)=(1,3)', '(3,2)=(2,3)']
Tổng số cặp được chốt: 6 = L*(L-1)/2 = 6 -> đủ mọi cặp ngoài đường chéo


## 6. Nền tảng 3 — Đệ quy và Backtracking (quay lui) từ căn bản

### 6.1 Ý tưởng

Insight ở mục 5 gợi ý cách làm: đặt hàng 0 → đặt hàng 1 → ... và **ngay khi** một hàng không có ứng viên nào thì **quay lui** (backtrack) để thử lựa chọn khác ở hàng trước. Đây chính là **backtracking**.

Backtracking thường được viết bằng **đệ quy** (recursion): một hàm tự gọi lại chính nó cho bài toán nhỏ hơn và có **điểm dừng** (base case). Khung chung gồm 3 động tác lặp lại:

```python
def backtrack(path):
    if path đã đầy đủ:            # điểm dừng
        ghi nhận bản sao path[:]
        return
    for lựa_chọn in các_lựa_chọn_hợp_lệ(path):
        path.append(lựa_chọn)      # 1. CHỌN
        backtrack(path)            # 2. ĐI TIẾP (đệ quy)
        path.pop()                 # 3. BỎ CHỌN (trả lại trạng thái cũ)
```

Ví dụ mini (chưa dính gì tới word square): sinh mọi chuỗi độ dài 2 từ các chữ `"ab"`, nhưng **cấm** hai chữ giống nhau đứng cạnh nhau. Cấm ngay khi chọn = **cắt nhánh** (pruning) sớm, giống hệt việc ta loại từ không có tiền tố đúng.

Cell sau in thụt lề theo độ sâu để bạn thấy đường "đi xuống" và "quay về".

In [10]:
def gen_no_repeat(letters, length):
    results, path = [], []

    def backtrack():
        pad = "  " * len(path)
        if len(path) == length:                       # điểm dừng
            print(pad + f"đủ dài -> ghi nhận {''.join(path)!r}")
            results.append("".join(path))             # lưu BẢN SAO (chuỗi mới)
            return
        for ch in letters:
            if path and path[-1] == ch:               # cắt nhánh: trùng chữ kề
                print(pad + f"bỏ qua {ch!r} (trùng chữ trước)")
                continue
            path.append(ch)                           # 1. chọn
            print(pad + f"chọn {ch!r} -> path={path}")
            backtrack()                               # 2. đi tiếp
            path.pop()                                # 3. bỏ chọn
    backtrack()
    return results

print(gen_no_repeat("ab", 2))

chọn 'a' -> path=['a']
  bỏ qua 'a' (trùng chữ trước)
  chọn 'b' -> path=['a', 'b']
    đủ dài -> ghi nhận 'ab'
chọn 'b' -> path=['b']
  chọn 'a' -> path=['b', 'a']
    đủ dài -> ghi nhận 'ba'
  bỏ qua 'b' (trùng chữ trước)
['ab', 'ba']


Đọc output:

- Từ `path = ['a']`, ta thử `'a'` → bị cắt ngay (trùng), thử `'b'` → đủ dài, ghi nhận `"ab"`.
- Sau khi `backtrack()` của `['a']` xong, dòng `path.pop()` trả `path` về `[]`, nhờ vậy vòng `for` ở mức ngoài cùng thử tiếp `'b'` từ một trạng thái **sạch**.

Ba điều cần nhớ khi viết backtracking (bài này dùng cả ba):

1. **Trạng thái dùng chung** (`path`) được sửa tại chỗ bằng `append` / `pop` — rẻ, không phải copy cả list mỗi bước.
2. Mỗi `append` phải đi kèm đúng một `pop` (chọn rồi phải bỏ chọn).
3. Khi ghi nhận kết quả phải lưu **bản sao** (`path[:]`), vì `path` sẽ còn bị `pop` tiếp (xem Lỗi 2).

## 7. Cách 2 — Backtracking + quét mọi từ bằng `startswith`

Ghép insight (mục 5) với khung backtracking (mục 6):

- `square` là `path`: list các hàng đã đặt.
- Điểm dừng: `len(square) == L` → ghi nhận `square[:]`.
- Lựa chọn hợp lệ cho hàng `r = len(square)`: mọi từ `w` thỏa `w.startswith(prefix)`.

`str.startswith(p)` trả `True` nếu chuỗi bắt đầu bằng `p`. Lưu ý đặc biệt: **mọi chuỗi đều bắt đầu bằng chuỗi rỗng** — `"area".startswith("")` là `True` — nên ở hàng 0 (tiền tố `""`) mọi từ đều là ứng viên, đúng như ta muốn, không cần viết `if` riêng.

In [11]:
print("'lead'.startswith('le') =", "lead".startswith("le"))
print("'lady'.startswith('le') =", "lady".startswith("le"))
print("'area'.startswith('')   =", "area".startswith(""))

def word_squares_scan(words):
    L = len(words[0])
    results, square = [], []

    def backtrack():
        r = len(square)
        if r == L:                                    # đủ L hàng
            results.append(square[:])
            return
        prefix = "".join(row[r] for row in square)    # cột r của các hàng đã đặt
        for w in words:                               # QUÉT mọi từ: O(N)
            if w.startswith(prefix):
                square.append(w)
                backtrack()
                square.pop()

    backtrack()
    return results

print(word_squares_scan(words1))
assert norm(word_squares_scan(words1)) == norm(expected1)
assert norm(word_squares_scan(words2)) == norm(expected2)
print("Cách 2: PASS cả 2 ví dụ")

'lead'.startswith('le') = True
'lady'.startswith('le') = False
'area'.startswith('')   = True
[['wall', 'area', 'lead', 'lady'], ['ball', 'area', 'lead', 'lady']]
Cách 2: PASS cả 2 ví dụ


### Nút thắt của cách 2

Cắt nhánh đã giúp ta **không** đi vào ngõ cụt như `"area"` ở hàng 0. Nhưng mỗi lần cần ứng viên cho một hàng, ta vẫn **quét cả `N` từ**, mỗi lần `startswith` tốn tới `O(L)` → `O(N·L)` cho **mỗi node** của cây tìm kiếm — kể cả khi chỉ có 0 hoặc 1 từ phù hợp.

Ta đo bằng cách đếm số lần gọi `startswith` và số lần thật sự "chọn" được một từ. Input "dày đặc" `dense4` = tất cả 16 chuỗi dài 4 trên bảng chữ `{a, b}`; vì mọi hình vuông 4x4 đối xứng trên `{a, b}` đều hợp lệ nên có tới `2^(4·5/2) = 2^10 = 1024` word square.

In [12]:
def count_scan_work(words):
    L = len(words[0])
    stats = {"startswith": 0, "chosen": 0, "found": 0}
    square = []

    def backtrack():
        r = len(square)
        if r == L:
            stats["found"] += 1
            return
        prefix = "".join(row[r] for row in square)
        for w in words:
            stats["startswith"] += 1
            if w.startswith(prefix):
                stats["chosen"] += 1
                square.append(w); backtrack(); square.pop()

    backtrack()
    return stats

dense4 = ["".join(t) for t in product("ab", repeat=4)]
print("dense4 có", len(dense4), "từ, ví dụ:", dense4[:5], "...")
for name, ws in [("words1", words1), ("words2", words2), ("dense4", dense4)]:
    s = count_scan_work(ws)
    print(f"{name:7}: gọi startswith {s['startswith']:6} lần, chọn được {s['chosen']:5} lần, "
          f"tìm thấy {s['found']} square -> lãng phí {s['startswith'] - s['chosen']} lần kiểm tra")

dense4 có 16 từ, ví dụ: ['aaaa', 'aaab', 'aaba', 'aabb', 'abaa'] ...
words1 : gọi startswith     55 lần, chọn được    12 lần, tìm thấy 2 square -> lãng phí 43 lần kiểm tra
words2 : gọi startswith     52 lần, chọn được    14 lần, tìm thấy 2 square -> lãng phí 38 lần kiểm tra
dense4 : gọi startswith  10512 lần, chọn được  1680 lần, tìm thấy 1024 square -> lãng phí 8832 lần kiểm tra


Phần lớn các lần gọi `startswith` trả `False` — công sức bỏ đi. Câu hỏi: có cách nào **hỏi thẳng** "những từ nào có tiền tố `'le'`?" mà nhận ngay danh sách, không phải quét?

Có — chuẩn bị trước một **bảng tra cứu** từ tiền tố sang danh sách từ. Đó là Hash Map (`dict`).

---

## 8. Nền tảng 4 — `dict`, `defaultdict(list)` và slicing tiền tố

### 8.1 `dict` và `defaultdict(list)`

`dict` lưu cặp **key → value**, tra cứu `key in d` hoặc `d.get(key)` nhanh **trung bình O(1)** (nhờ hàm băm — hashing). Ta muốn: key = tiền tố, value = **list** các từ có tiền tố đó.

Với `dict` thường, trước khi `append` vào list của một key mới ta phải tạo list rỗng, rất dài dòng. `collections.defaultdict(list)` tự tạo `[]` cho key chưa có ngay khi ta truy cập `d[key]`.

Cẩn thận: chính vì "tự tạo", nếu ta chỉ muốn **đọc** thì nên dùng `d.get(key, [])` — vì `d[key]` sẽ âm thầm **thêm** key rỗng vào bảng.

In [13]:
from collections import defaultdict

normal = {}
if "l" not in normal:          # dict thường: phải tự tạo list
    normal["l"] = []
normal["l"].append("lead")
print("dict thường     :", normal)

groups = defaultdict(list)     # defaultdict: tự tạo [] cho key mới
groups["l"].append("lead")
groups["l"].append("lady")
groups["w"].append("wall")
print("defaultdict     :", dict(groups))

print("get key có      :", groups.get("l", []))
print("get key không có:", groups.get("x", []), "| số key vẫn là", len(groups))
_ = groups["x"]                # ĐỌC bằng [] -> tự thêm key 'x'!
print("sau groups['x'] :", dict(groups), "| số key thành", len(groups))

dict thường     : {'l': ['lead']}
defaultdict     : {'l': ['lead', 'lady'], 'w': ['wall']}
get key có      : ['lead', 'lady']
get key không có: [] | số key vẫn là 2
sau groups['x'] : {'l': ['lead', 'lady'], 'w': ['wall'], 'x': []} | số key thành 3


### 8.2 Slicing lấy tiền tố: `w[:k]`

`w[:k]` lấy `k` ký tự đầu của `w` (từ chỉ số 0 tới trước chỉ số `k`). Đặc biệt `w[:0] == ""` (chuỗi rỗng).

Tiền tố ta cần tra luôn có độ dài `r` với `0 <= r <= L-1` (hàng `r` chạy từ `0` tới `L-1`). Vì vậy với mỗi từ, ta lưu nó vào các key `w[:0], w[:1], ..., w[:L-1]` — tổng cộng `L` key mỗi từ. Không cần lưu tiền tố dài `L` (chính là cả từ), vì không bao giờ có hàng nào cần tra tiền tố dài `L`.

Nhờ lưu cả key `""` → tất cả các từ, hàng 0 cũng được xử lý bằng **đúng một** dòng tra cứu như mọi hàng khác.

In [14]:
w = "lead"
for k in range(len(w)):
    print(f"w[:{k}] = {w[:k]!r}")

def build_prefix_map(words):
    L = len(words[0])
    prefix_map = defaultdict(list)
    for w in words:
        for k in range(L):                 # tiền tố dài 0..L-1
            prefix_map[w[:k]].append(w)
    return prefix_map

pm1 = build_prefix_map(words1)
for key in sorted(pm1, key=lambda s: (len(s), s)):
    print(f"  {key!r:7} -> {pm1[key]}")

w[:0] = ''
w[:1] = 'l'
w[:2] = 'le'
w[:3] = 'lea'
  ''      -> ['area', 'lead', 'wall', 'lady', 'ball']
  'a'     -> ['area']
  'b'     -> ['ball']
  'l'     -> ['lead', 'lady']
  'w'     -> ['wall']
  'ar'    -> ['area']
  'ba'    -> ['ball']
  'la'    -> ['lady']
  'le'    -> ['lead']
  'wa'    -> ['wall']
  'are'   -> ['area']
  'bal'   -> ['ball']
  'lad'   -> ['lady']
  'lea'   -> ['lead']
  'wal'   -> ['wall']


## 9. Cách 3 (cách chính) — Backtracking + Hash Map tiền tố

So với cách 2, ta chỉ thay **đúng một dòng**: thay vì

```python
for w in words:
    if w.startswith(prefix): ...
```

ta viết

```python
for w in prefix_map.get(prefix, []): ...
```

Toàn bộ thuật toán gồm 3 đoạn:

1. **Tiền xử lý**: `prefix_map = build_prefix_map(words)` — mỗi từ được thêm vào `L` danh sách.
2. **`backtrack()`**: gọi `r = len(square)`.
   - Nếu `r == L` → ghi nhận `square[:]`.
   - Ngược lại tính `prefix` = cột `r` của các hàng đã đặt, rồi thử từng `w` trong `prefix_map.get(prefix, [])`: chọn → đi tiếp → bỏ chọn.
3. Gọi `backtrack()` một lần từ `square = []` (hàng 0 dùng tiền tố `""` = mọi từ).

Nếu tiền tố không có trong bảng, `get` trả `[]`, vòng `for` không chạy lần nào → hàm tự `return` → **quay lui** ngay. Đó chính là cắt nhánh, không cần viết `if` riêng.

In [15]:
def word_squares_hashmap(words):
    L = len(words[0])
    prefix_map = build_prefix_map(words)          # 1. tiền xử lý
    results, square = [], []

    def backtrack():                              # 2. quay lui
        r = len(square)
        if r == L:
            results.append(square[:])             #    lưu BẢN SAO
            return
        prefix = "".join(row[r] for row in square)
        for w in prefix_map.get(prefix, []):      #    chỉ các từ có đúng tiền tố
            square.append(w)                      #    chọn
            backtrack()                           #    đi tiếp
            square.pop()                          #    bỏ chọn

    backtrack()                                   # 3. bắt đầu từ hình vuông rỗng
    return results

r1, r2 = word_squares_hashmap(words1), word_squares_hashmap(words2)
print(r1)
print(r2)
assert norm(r1) == norm(expected1)
assert norm(r2) == norm(expected2)
print("Cách 3: PASS cả 2 ví dụ")

[['wall', 'area', 'lead', 'lady'], ['ball', 'area', 'lead', 'lady']]
[['baba', 'abat', 'baba', 'atan'], ['baba', 'abat', 'baba', 'atal']]
Cách 3: PASS cả 2 ví dụ


## 10. Trace cách chính trên Example 1 `["area","lead","wall","lady","ball"]`

Bảng trạng thái theo thứ tự thăm (mỗi dòng là một lần gọi `backtrack()`; `square` là trạng thái **trước** khi chọn hàng `r`):

| Bước | `square` hiện tại | `r` | Tiền tố bắt buộc (cột `r`) | `prefix_map.get(...)` | Quyết định |
|---:|---|---:|---|---|---|
| 1 | `[]` | 0 | `""` | `[area, lead, wall, lady, ball]` | thử lần lượt 5 từ |
| 2 | `[area]` | 1 | `area[1]` = `"r"` | `[]` | ngõ cụt → bỏ `area` |
| 3 | `[lead]` | 1 | `lead[1]` = `"e"` | `[]` | ngõ cụt → bỏ `lead` |
| 4 | `[wall]` | 1 | `"a"` | `[area]` | chọn `area` |
| 5 | `[wall, area]` | 2 | `wall[2] + area[2]` = `"le"` | `[lead]` | chọn `lead` |
| 6 | `[wall, area, lead]` | 3 | `"l" + "a" + "d"` = `"lad"` | `[lady]` | chọn `lady` |
| 7 | `[wall, area, lead, lady]` | 4 | — | — | `r == L` → **ghi nhận** |
| 8 | `[lady]` | 1 | `"a"` | `[area]` | chọn `area` |
| 9 | `[lady, area]` | 2 | `lady[2] + area[2]` = `"de"` | `[]` | ngõ cụt → quay lui 2 lần |
| 10 | `[ball]` | 1 | `"a"` | `[area]` | chọn `area` |
| 11 | `[ball, area]` | 2 | `"le"` | `[lead]` | chọn `lead` |
| 12 | `[ball, area, lead]` | 3 | `"lad"` | `[lady]` | chọn `lady` |
| 13 | `[ball, area, lead, lady]` | 4 | — | — | `r == L` → **ghi nhận** |

Ý nghĩa các cột: "Tiền tố bắt buộc" luôn lấy ký tự **thứ `r`** của từng hàng đã đặt (đọc **dọc** cột `r`). "Quyết định" cho biết ta đi sâu xuống hay quay lui.

Hàm `trace_word_squares` in đúng quá trình này (thụt lề theo độ sâu `r`):

In [16]:
def trace_word_squares(words):
    L = len(words[0])
    prefix_map = build_prefix_map(words)
    results, square = [], []
    calls = [0]

    def backtrack():
        calls[0] += 1
        r = len(square)
        pad = "    " * r
        if r == L:
            print(pad + f"[{calls[0]:2}] r={r} == L -> GHI NHẬN {square}")
            results.append(square[:])
            return
        prefix = "".join(row[r] for row in square)
        cands = prefix_map.get(prefix, [])
        note = "  -> NGÕ CỤT, quay lui" if not cands else ""
        print(pad + f"[{calls[0]:2}] r={r}, square={square}, tiền tố={prefix!r}, ứng viên={cands}{note}")
        for w in cands:
            square.append(w)
            backtrack()
            square.pop()

    backtrack()
    print("Kết quả:", results)
    print("Số lần gọi backtrack:", calls[0])
    return results

_ = trace_word_squares(words1)

[ 1] r=0, square=[], tiền tố='', ứng viên=['area', 'lead', 'wall', 'lady', 'ball']
    [ 2] r=1, square=['area'], tiền tố='r', ứng viên=[]  -> NGÕ CỤT, quay lui
    [ 3] r=1, square=['lead'], tiền tố='e', ứng viên=[]  -> NGÕ CỤT, quay lui
    [ 4] r=1, square=['wall'], tiền tố='a', ứng viên=['area']
        [ 5] r=2, square=['wall', 'area'], tiền tố='le', ứng viên=['lead']
            [ 6] r=3, square=['wall', 'area', 'lead'], tiền tố='lad', ứng viên=['lady']
                [ 7] r=4 == L -> GHI NHẬN ['wall', 'area', 'lead', 'lady']
    [ 8] r=1, square=['lady'], tiền tố='a', ứng viên=['area']
        [ 9] r=2, square=['lady', 'area'], tiền tố='de', ứng viên=[]  -> NGÕ CỤT, quay lui
    [10] r=1, square=['ball'], tiền tố='a', ứng viên=['area']
        [11] r=2, square=['ball', 'area'], tiền tố='le', ứng viên=['lead']
            [12] r=3, square=['ball', 'area', 'lead'], tiền tố='lad', ứng viên=['lady']
                [13] r=4 == L -> GHI NHẬN ['ball', 'area', 'lead', 'lady']
Kết quả:

Đối chiếu với bảng: 13 lần gọi `backtrack()` khớp 13 bước, hai lần **ghi nhận** ở bước 7 và 13, ba ngõ cụt ở bước 2, 3, 9.

Quan sát đáng giá:

- Với `area` và `lead` ở hàng 0, ta dừng **ngay ở hàng 1** — brute force thì sẽ thử thêm `5^3 = 125` bộ cho mỗi từ.
- Với `lady` ở hàng 0, ta đi được 1 hàng rồi mới gãy ở hàng 2 (tiền tố `"de"`).
- Kết quả trả ra theo thứ tự `wall...` trước `ball...` (vì `wall` đứng trước `ball` trong `words`), ngược với thứ tự trong đề — hoàn toàn hợp lệ vì đề chấp nhận mọi thứ tự.

Thử thêm Example 2 để thấy một từ (`baba`) được dùng **hai lần** — Hash Map trả lại `baba` ở hàng 2 dù nó đã nằm ở hàng 0, vì ta không hề cấm dùng lại.

In [17]:
_ = trace_word_squares(words2)

[ 1] r=0, square=[], tiền tố='', ứng viên=['abat', 'baba', 'atan', 'atal']
    [ 2] r=1, square=['abat'], tiền tố='b', ứng viên=['baba']
        [ 3] r=2, square=['abat', 'baba'], tiền tố='ab', ứng viên=['abat']
            [ 4] r=3, square=['abat', 'baba', 'abat'], tiền tố='tat', ứng viên=[]  -> NGÕ CỤT, quay lui
    [ 5] r=1, square=['baba'], tiền tố='a', ứng viên=['abat', 'atan', 'atal']
        [ 6] r=2, square=['baba', 'abat'], tiền tố='ba', ứng viên=['baba']
            [ 7] r=3, square=['baba', 'abat', 'baba'], tiền tố='ata', ứng viên=['atan', 'atal']
                [ 8] r=4 == L -> GHI NHẬN ['baba', 'abat', 'baba', 'atan']
                [ 9] r=4 == L -> GHI NHẬN ['baba', 'abat', 'baba', 'atal']
        [10] r=2, square=['baba', 'atan'], tiền tố='ba', ứng viên=['baba']
            [11] r=3, square=['baba', 'atan', 'baba'], tiền tố='ana', ứng viên=[]  -> NGÕ CỤT, quay lui
        [12] r=2, square=['baba', 'atal'], tiền tố='ba', ứng viên=['baba']
            [13] r=3, square=['

### So sánh khối lượng công việc: quét (cách 2) và Hash Map (cách 3)

Cả hai cách đi qua **cùng một cây tìm kiếm** (cùng số node, cùng kết quả). Khác biệt nằm ở chi phí **mỗi node**: cách 2 kiểm tra `N` từ, cách 3 chỉ duyệt đúng các từ ứng viên. Ta đếm "số từ được xem xét" của cách 3 và so với số lần `startswith` của cách 2, rồi đo thời gian trên một input ngẫu nhiên lớn (`random.seed` cố định để kết quả lặp lại được).

In [18]:
import random, time, string

def count_hashmap_work(words):
    L = len(words[0]); pm = build_prefix_map(words)
    stats = {"looked": 0, "found": 0}
    square = []
    def backtrack():
        r = len(square)
        if r == L:
            stats["found"] += 1; return
        for w in pm.get("".join(row[r] for row in square), []):
            stats["looked"] += 1
            square.append(w); backtrack(); square.pop()
    backtrack()
    return stats

random.seed(425)
big = sorted({"".join(random.choices("abcdef", k=4)) for _ in range(300)})
for name, ws in [("words1", words1), ("dense4", dense4), (f"big({len(big)} từ)", big)]:
    a, b = count_scan_work(ws), count_hashmap_work(ws)
    assert a["found"] == b["found"]
    print(f"{name:13}: cách 2 xét {a['startswith']:8} từ | cách 3 xét {b['looked']:6} từ | {b['found']} square")

for fn in (word_squares_scan, word_squares_hashmap):
    t = time.perf_counter(); out = fn(big); t = time.perf_counter() - t
    print(f"{fn.__name__:22}: {len(out)} square, {t*1000:8.1f} ms")

words1       : cách 2 xét       55 từ | cách 3 xét     12 từ | 2 square
dense4       : cách 2 xét    10512 từ | cách 3 xét   1680 từ | 1024 square


big(261 từ)  : cách 2 xét 24494589 từ | cách 3 xét 192186 từ | 98338 square


word_squares_scan     : 98338 square,    504.2 ms
word_squares_hashmap  : 98338 square,    143.2 ms


## 11. Nền tảng 5 — Trie (cây tiền tố) từ căn bản

Hash Map lưu **mọi tiền tố dưới dạng chuỗi riêng**: `"l"`, `"le"`, `"lea"` là 3 key khác nhau, dù `"lea"` "chứa" sẵn `"le"`. Một cấu trúc tận dụng việc các tiền tố **lồng nhau** là **Trie** (đọc là "try", còn gọi prefix tree — cây tiền tố).

- Mỗi **node** đại diện cho một tiền tố. **Root** (gốc) là tiền tố rỗng `""`.
- Mỗi node có `children`: một `dict` ký tự → node con. Đi từ root theo các ký tự `l → e` ta tới node của tiền tố `"le"`.
- Các từ chung tiền tố **dùng chung** các node đầu: `lead` và `lady` chung node `l`.

Ví dụ mini: Trie "trơn" (chỉ đánh dấu `is_end` = có từ kết thúc ở đây) với 3 thao tác: `insert`, `search` (có đúng từ này không), `starts_with` (có từ nào bắt đầu bằng tiền tố này không). `dict.setdefault(ch, TrieNodeBasic())` nghĩa là: nếu đã có con `ch` thì lấy nó, chưa có thì tạo mới rồi lấy.

In [19]:
class TrieNodeBasic:
    def __init__(self):
        self.children = {}      # ký tự -> TrieNodeBasic
        self.is_end = False     # có từ nào kết thúc đúng tại node này?

def trie_insert(root, word):
    node = root
    for ch in word:
        node = node.children.setdefault(ch, TrieNodeBasic())
    node.is_end = True

def trie_walk(root, s):
    node = root
    for ch in s:
        node = node.children.get(ch)
        if node is None:
            return None         # gãy giữa đường: không có tiền tố này
    return node

def print_trie(node, indent=""):
    items = sorted(node.children.items())
    for i, (ch, child) in enumerate(items):
        last = (i == len(items) - 1)
        star = " ★" if getattr(child, "is_end", False) else ""
        extra = f"   words={child.words}" if hasattr(child, "words") else ""
        print(indent + ("└── " if last else "├── ") + ch + star + extra)
        print_trie(child, indent + ("    " if last else "│   "))

root_basic = TrieNodeBasic()
for w in ["lead", "lady", "lea"]:
    trie_insert(root_basic, w)
print("(root)")
print_trie(root_basic)
node = trie_walk(root_basic, "lea")
print("search('lea')      :", node is not None and node.is_end)
node = trie_walk(root_basic, "la")
print("search('la')       :", node is not None and node.is_end)
print("starts_with('la')  :", trie_walk(root_basic, "la") is not None)
print("starts_with('lo')  :", trie_walk(root_basic, "lo") is not None)

(root)
└── l
    ├── a
    │   └── d
    │       └── y ★
    └── e
        └── a ★
            └── d ★
search('lea')      : True
search('la')       : False
starts_with('la')  : True
starts_with('lo')  : False


### 11.1 Trie cho bài này: mỗi node lưu **danh sách các từ đi qua nó**

Bài toán không hỏi "có từ nào bắt đầu bằng `p` không" mà hỏi "**những từ nào** bắt đầu bằng `p`". Ta cho mỗi node thêm thuộc tính `words`: list các từ **đi qua** node đó (tức là có tiền tố tương ứng). Khi chèn một từ, ta `append` nó vào `words` của **mọi node trên đường đi**, kể cả root.

Tra cứu tiền tố `prefix`: đi từ root theo từng ký tự (`O(len(prefix))`); gãy giữa đường → `[]`; đi hết → trả `node.words`.

Giống Hash Map, ta chỉ cần lưu vào các node có độ sâu `0..L-1` — nhưng để code đơn giản, cell dưới chèn trọn từ (node độ sâu `L` cũng được tạo, chỉ là không bao giờ được tra tới).

In [20]:
class TrieNode:
    def __init__(self):
        self.children = {}      # ký tự -> TrieNode
        self.words = []         # các từ có tiền tố = đường đi từ root tới đây

def build_trie(words):
    root = TrieNode()
    for w in words:
        node = root
        node.words.append(w)                  # root: tiền tố "" -> mọi từ
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
            node.words.append(w)
    return root

def find_by_prefix(root, prefix):
    node = root
    for ch in prefix:
        node = node.children.get(ch)
        if node is None:
            return []
    return node.words

trie1 = build_trie(["area", "lead", "lady"])     # 3 từ cho cây gọn
print("(root) words =", trie1.words)
print_trie(trie1)
print("find 'l'  :", find_by_prefix(trie1, "l"))
print("find 'le' :", find_by_prefix(trie1, "le"))
print("find 'de' :", find_by_prefix(trie1, "de"))

(root) words = ['area', 'lead', 'lady']
├── a   words=['area']
│   └── r   words=['area']
│       └── e   words=['area']
│           └── a   words=['area']
└── l   words=['lead', 'lady']
    ├── a   words=['lady']
    │   └── d   words=['lady']
    │       └── y   words=['lady']
    └── e   words=['lead']
        └── a   words=['lead']
            └── d   words=['lead']
find 'l'  : ['lead', 'lady']
find 'le' : ['lead']
find 'de' : []


## 12. Cách 4 — Backtracking + Trie

Khung backtracking giữ nguyên, chỉ thay nguồn ứng viên: `prefix_map.get(prefix, [])` → `find_by_prefix(root, prefix)`.

Một cải tiến nhỏ đáng biết: thay vì nối chuỗi `prefix` rồi đi lại từ root mỗi lần, ta có thể đi từ root theo từng ký tự `square[j][r]` ngay lúc tính — gãy ở đâu dừng ở đó, không tạo chuỗi tạm. Đó là việc của vòng `for` trong `candidates(r)` dưới đây.

In [21]:
def word_squares_trie(words):
    L = len(words[0])
    root = build_trie(words)
    results, square = [], []

    def candidates(r):
        node = root
        for row in square:                    # đi dọc cột r: square[0][r], square[1][r], ...
            node = node.children.get(row[r])
            if node is None:
                return []                     # gãy -> không có từ nào
        return node.words

    def backtrack():
        r = len(square)
        if r == L:
            results.append(square[:])
            return
        for w in candidates(r):
            square.append(w)
            backtrack()
            square.pop()

    backtrack()
    return results

print(word_squares_trie(words1))
assert norm(word_squares_trie(words1)) == norm(expected1)
assert norm(word_squares_trie(words2)) == norm(expected2)
print("Cách 4: PASS cả 2 ví dụ")

[['wall', 'area', 'lead', 'lady'], ['ball', 'area', 'lead', 'lady']]
Cách 4: PASS cả 2 ví dụ


### 12.1 Hash Map hay Trie? Đo bộ nhớ và thời gian

Cả hai đều lưu mỗi từ vào khoảng `L` danh sách (tổng `≈ N·L` tham chiếu tới từ). Khác nhau ở phần "khung":

- Hash Map: mỗi tiền tố là một **chuỗi key riêng**; tổng số ký tự trong các key có thể tới `N·(0+1+...+(L-1)) = O(N·L^2)`.
- Trie: mỗi tiền tố là **một node**, ký tự chung được dùng chung; số node `<= N·L + 1`.

Với `L <= 4` khác biệt không lớn. Về tốc độ, hai cách xấp xỉ nhau: Hash Map phải **nối chuỗi** `prefix` rồi băm nó, còn Trie đi `r` bước qua các node nhưng không tạo chuỗi tạm — con số cụ thể tùy máy, đừng coi cách nào "luôn nhanh hơn". Đo trên 1000 từ ngẫu nhiên (đúng cỡ constraints `N = 1000`, `L = 4`):

In [22]:
def count_trie_nodes(node):
    return 1 + sum(count_trie_nodes(ch) for ch in node.children.values())

random.seed(2024)
pool1000 = set()
while len(pool1000) < 1000:                       # đúng 1000 từ khác nhau, dài 4, chữ a..z
    pool1000.add("".join(random.choices(string.ascii_lowercase, k=4)))
big1000 = sorted(pool1000)
pm_big, trie_big = build_prefix_map(big1000), build_trie(big1000)
print("N =", len(big1000))
print("Hash Map: số key =", len(pm_big), "| tổng ký tự trong key =", sum(len(k) for k in pm_big),
      "| tổng phần tử trong các list =", sum(len(v) for v in pm_big.values()))
print("Trie    : số node =", count_trie_nodes(trie_big))

outs = {}
for fn in (word_squares_hashmap, word_squares_trie):
    t = time.perf_counter(); outs[fn.__name__] = fn(big1000); t = time.perf_counter() - t
    print(f"{fn.__name__:22}: {len(outs[fn.__name__])} square, {t*1000:8.1f} ms")
assert norm(outs["word_squares_hashmap"]) == norm(outs["word_squares_trie"])

N = 1000
Hash Map: số key = 1509 | tổng ký tự trong key = 3954 | tổng phần tử trong các list = 4000
Trie    : số node = 2509


word_squares_hashmap  : 3268 square,     50.9 ms
word_squares_trie     : 3268 square,     24.1 ms


## 13. Code nộp LeetCode — cách chính (Backtracking + Hash Map tiền tố)

Chuyển `word_squares_hashmap` vào đúng signature `wordSquares(self, words: List[str]) -> List[List[str]]`:

- Dùng `defaultdict(list)` (nhớ `from collections import defaultdict`; LeetCode đã sẵn `List`).
- Hàm lồng `backtrack` dùng `square`, `prefix_map`, `results` của hàm ngoài (closure) — không cần truyền tham số.
- Tra cứu bằng `.get(prefix, [])` để không vô tình thêm key rỗng vào bảng.

In [23]:
from collections import defaultdict
from typing import List

class Solution:
    def wordSquares(self, words: List[str]) -> List[List[str]]:
        L = len(words[0])
        prefix_map = defaultdict(list)
        for w in words:
            for k in range(L):                       # tiền tố dài 0..L-1
                prefix_map[w[:k]].append(w)

        results, square = [], []

        def backtrack():
            r = len(square)
            if r == L:
                results.append(square[:])
                return
            prefix = "".join(row[r] for row in square)   # cột r của các hàng đã đặt
            for w in prefix_map.get(prefix, []):
                square.append(w)
                backtrack()
                square.pop()

        backtrack()
        return results

print(Solution().wordSquares(words1))
print(Solution().wordSquares(words2))

[['wall', 'area', 'lead', 'lady'], ['ball', 'area', 'lead', 'lady']]
[['baba', 'abat', 'baba', 'atan'], ['baba', 'abat', 'baba', 'atal']]


### Bộ test

Ngoài 2 ví dụ của đề, ta thêm các edge case:

- `L = 1`: mỗi từ 1 chữ tự nó là một hình vuông `1x1` → mỗi từ là một đáp án.
- Một từ duy nhất `["abba"]`: `abba` không thể làm hàng 1 (cần bắt đầu bằng `'b'`) → `[]`.
- Từ "tự đối xứng" dùng lại chính nó: `["aa"]` → `[["aa","aa"]]` (cùng một từ ở cả 2 hàng).
- `dense4`: 16 từ trên `{a, b}` → đúng `1024` hình vuông, và mọi hình vuông đều qua trọng tài `is_word_square`.
- Không có hình vuông nào: `["abc", "def"]` → `[]`.

Sau đó (cell thứ hai) **đối chiếu chéo 5 cách** trên input ngẫu nhiên nhỏ: sinh (với `random.seed` cố định) các bộ từ không trùng, độ dài `L` từ 1 tới 3 trên bảng chữ nhỏ `{a, b, c}` (bảng chữ nhỏ thì mới hay có word square). Brute force là đáp án chuẩn; 4 cách còn lại phải cho **cùng tập kết quả**.

In [24]:
sol = Solution()

def check(name, got, expected):
    assert norm(got) == norm(expected), (name, got, expected)
    print(f"{name:28} PASS")

check("Example 1", sol.wordSquares(words1), expected1)
check("Example 2", sol.wordSquares(words2), expected2)
check("L = 1", sol.wordSquares(["a", "b", "z"]), [["a"], ["b"], ["z"]])
check("một từ không tạo được", sol.wordSquares(["abba"]), [])
check("dùng lại chính nó", sol.wordSquares(["aa"]), [["aa", "aa"]])
check("không có square", sol.wordSquares(["abc", "def"]), [])

dense_out = sol.wordSquares(dense4)
assert len(dense_out) == 1024 and all(is_word_square(sq) for sq in dense_out)
assert len(set(map(tuple, dense_out))) == 1024          # không trùng lặp
print(f"{'dense4: 1024 square hợp lệ':28} PASS")

Example 1                    PASS
Example 2                    PASS
L = 1                        PASS
một từ không tạo được        PASS
dùng lại chính nó            PASS
không có square              PASS
dense4: 1024 square hợp lệ   PASS


In [25]:
random.seed(425)
total_squares = 0
for trial in range(300):
    L = random.randint(1, 3)
    pool = ["".join(t) for t in product("abc", repeat=L)]
    words = random.sample(pool, random.randint(1, min(8, len(pool))))
    ref = norm(word_squares_bruteforce(words))
    for fn in (word_squares_scan, word_squares_hashmap, word_squares_trie, Solution().wordSquares):
        assert norm(fn(words)) == ref, (fn, words)
    total_squares += len(ref)
print("300 bộ test ngẫu nhiên: 5 cách khớp nhau. Tổng số square tìm được:", total_squares)

300 bộ test ngẫu nhiên: 5 cách khớp nhau. Tổng số square tìm được: 1540


## 14. Vì sao cách chính đúng?

**Invariant** (bất biến) — điều luôn đúng mỗi khi vào `backtrack()` với `square` có `r` hàng:

> `square[0..r-1]` là một "word square một phần": với mọi `i, j < r` thì `square[i][j] == square[j][i]`.

- **Ban đầu** `r = 0`: hiển nhiên đúng (không có cặp nào).
- **Duy trì**: ta chỉ thêm hàng `w` khi `w` bắt đầu bằng `prefix = square[0][r] + ... + square[r-1][r]`, tức là `w[j] == square[j][r]` với mọi `j < r` — đúng các cặp mới xuất hiện khi thêm hàng `r` (mục 5.1). Các cặp cũ không đổi. `pop()` trả `square` về đúng trạng thái trước, nên invariant vẫn đúng cho lựa chọn kế tiếp.
- **Kết thúc** `r == L`: mọi cặp đều thỏa → hình vuông hợp lệ. Vậy **không trả nhầm**.

**Không bỏ sót**: lấy một word square hợp lệ bất kỳ `S = [s0, ..., s_{L-1}]`. Ở hàng 0, `s0` nằm trong `prefix_map[""]` (mọi từ). Giả sử đã đặt đúng `s0..s_{r-1}`; vì `S` hợp lệ nên `s_r` bắt đầu bằng cột `r` của các hàng trên — chính là `prefix` ta tính → `s_r` nằm trong `prefix_map[prefix]` và sẽ được thử. Quy nạp → `S` được ghi nhận.

**Không trùng lặp**: mỗi đường đi từ gốc tới lá tương ứng một dãy lựa chọn khác nhau; trong mỗi danh sách ứng viên các từ khác nhau (vì `words` không trùng) → hai lá khác nhau cho hai dãy khác nhau.

## 15. Độ phức tạp

Gọi `N` = số từ, `L` = độ dài từ.

**Tiền xử lý** (xây `prefix_map`): mỗi từ tạo `L` tiền tố, mỗi tiền tố cắt tốn `O(L)` → **`O(N·L^2)` time**, bộ nhớ `O(N·L^2)` (key) + `O(N·L)` (các list).

**Tìm kiếm**: mỗi node của cây tốn `O(L)` để nối `prefix` + `O(L)` trung bình để băm và tra `dict`, và mỗi lá tốn `O(L)` để copy `square[:]`. Vậy thời gian = `O(số node của cây · L)`.

Số node bị chặn như sau: hàng 0 có tối đa `N` lựa chọn; hàng `r >= 1` đã cố định `r` chữ đầu, nên còn tối đa `min(N, 26^(L-r))` từ có tiền tố đó. Do đó số lá (và số đáp án) nhiều nhất là

$$N \cdot \prod_{r=1}^{L-1} \min\left(N,\ 26^{L-r}\right).$$

Đây là chặn trên rất thô cho trường hợp xấu nhất; trên thực tế ứng viên giảm cực nhanh theo `r` (cell dưới, 1000 từ ngẫu nhiên: mỗi lựa chọn hàng 0 cho trung bình ~38 ứng viên ở hàng 1, ~1.5 ở hàng 2 và chỉ ~0.06 ở hàng 3). Lưu ý thời gian **không thể** dưới kích thước output: `dense4` có 1024 đáp án thì phải in 1024·4 từ.

**Bộ nhớ phụ**: `prefix_map` `O(N·L^2)`; đệ quy sâu tối đa `L` tầng + `square` dài `L` → `O(L)`; chưa tính output.

So với brute force `O(N^L · L^2)`: brute force duyệt **mọi** bộ, còn backtracking chỉ đi vào những nhánh mà tiền tố vẫn còn "sống".

In [26]:
def count_nodes(words):
    L = len(words[0]); pm = build_prefix_map(words)
    per_level = [0] * (L + 1); square = []
    def backtrack():
        r = len(square); per_level[r] += 1
        if r == L: return
        for w in pm.get("".join(row[r] for row in square), []):
            square.append(w); backtrack(); square.pop()
    backtrack()
    return per_level

for name, ws in [("words1", words1), ("dense4", dense4), ("big1000", big1000)]:
    lv = count_nodes(ws)
    print(f"{name:8} N={len(ws):4}: số node theo độ sâu r=0..L: {lv} | tổng {sum(lv)} "
          f"vs brute force N^L = {len(ws) ** len(ws[0]):,}")

words1   N=   5: số node theo độ sâu r=0..L: [1, 5, 3, 2, 2] | tổng 13 vs brute force N^L = 625
dense4   N=  16: số node theo độ sâu r=0..L: [1, 16, 128, 512, 1024] | tổng 1681 vs brute force N^L = 65,536


big1000  N=1000: số node theo độ sâu r=0..L: [1, 1000, 38550, 56921, 3268] | tổng 99740 vs brute force N^L = 1,000,000,000,000


Đọc output: với `big1000` (1000 từ ngẫu nhiên trên 26 chữ cái), brute force phải thử `10^12` bộ, còn cây backtracking chỉ có khoảng `10^5` node. Từ độ sâu 1 sang 2 số node tăng ~38 lần (tiền tố 1 chữ khớp nhiều từ), sang độ sâu 3 chỉ tăng ~1.5 lần, và sang độ sâu 4 thì **giảm** mạnh (tiền tố 3 chữ hiếm khi khớp). Với `dense4` thì ngược lại: gần như nhánh nào cũng sống, vì mọi hình vuông đối xứng đều hợp lệ — đó là kiểu input "xấu" cho backtracking, và số node bị chặn bởi kích thước output.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Nhận xét |
|---|---|---|---|---|
| 1. Brute force | `product(words, repeat=L)` + kiểm tra | `O(N^L · L^2)` | `O(L)` | Đúng, dễ viết; chỉ dùng làm đáp án chuẩn trên input nhỏ |
| 2. Backtracking + quét | tiền tố bắt buộc, quét `N` từ với `startswith` | `O(số node · N · L)` | `O(L)` | Cắt nhánh sớm nhưng mỗi node vẫn tốn `O(N·L)` |
| 3. Backtracking + Hash Map ★ | `prefix_map[prefix]` → danh sách ứng viên | `O(N·L^2 + số node · L)` | `O(N·L^2)` | **Cách chính**: code ngắn nhất, dễ viết đúng; tốc độ ngang Trie |
| 4. Backtracking + Trie | node lưu `words`, đi dọc cột `r` | `O(N·L + số node · L)` | `O(N·L)` | Tiết kiệm phần key, không tạo chuỗi tạm; hợp khi từ dài; code dài hơn |

("số node" = số lần gọi `backtrack()`, giống nhau ở cách 2, 3, 4 vì cùng một cây tìm kiếm.)

## 17. Lỗi thường gặp

### Lỗi 1: Dùng `permutations` (hoặc cấm dùng lại từ) — mất đáp án

Người mới hay nghĩ "chọn 4 từ trong danh sách" = hoán vị. Nhưng đề cho phép **dùng lại** một từ. Example 2 cần `baba` hai lần, nên brute force bằng `permutations` trả `[]`. Đáng sợ hơn: Example 1 **vẫn đúng** (hai đáp án dùng 4 từ khác nhau), nên nếu chỉ test Example 1 bạn sẽ không phát hiện.

Sửa: dùng `product(words, repeat=L)`; trong backtracking thì **không** đánh dấu `used`.

In [27]:
def wrong_permutations(words):
    L = len(words[0])
    return [list(c) for c in permutations(words, L) if is_word_square(c)]

print("Example 1:", norm(wrong_permutations(words1)) == norm(expected1), "(vẫn đúng -> dễ bị lừa)")
print("Example 2:", wrong_permutations(words2), "<- SAI, mong đợi 2 square")

Example 1: True (vẫn đúng -> dễ bị lừa)
Example 2: [] <- SAI, mong đợi 2 square


### Lỗi 2: `results.append(square)` — quên lưu **bản sao**

`square` là **một** list duy nhất được sửa liên tục. `results.append(square)` chỉ lưu **tham chiếu** (cái "nhãn" trỏ tới list đó), nên mọi phần tử của `results` là **cùng một** list. Sau khi backtracking kết thúc, mọi `pop()` đã làm `square` rỗng → kết quả là `[[], []]`.

Sửa: `results.append(square[:])` (hoặc `list(square)`) để chụp lại trạng thái tại thời điểm ghi nhận.

In [28]:
def wrong_no_copy(words):
    L = len(words[0]); pm = build_prefix_map(words)
    results, square = [], []
    def backtrack():
        r = len(square)
        if r == L:
            results.append(square)                # SAI: lưu tham chiếu
            return
        for w in pm.get("".join(row[r] for row in square), []):
            square.append(w); backtrack(); square.pop()
    backtrack()
    return results

bad = wrong_no_copy(words1)
print("Kết quả sai:", bad)
print("Hai phần tử là cùng một object?", bad[0] is bad[1])

Kết quả sai: [[], []]
Hai phần tử là cùng một object? True


### Lỗi 3: Tính tiền tố theo **hàng** thay vì theo **cột**

Một nhầm lẫn rất tự nhiên: "hàng mới phải nối tiếp hàng trước", nên viết `prefix = square[-1][:r]` (lấy `r` chữ đầu của hàng ngay trên). Nhưng tiền tố đúng là **cột `r`** của **tất cả** các hàng trên: `"".join(row[r] for row in square)`.

Với Example 1, bản sai sinh ra những "hình vuông" như `["area","area","area","area"]` hay `["ball","ball","ball","ball"]` — cột 0 của cái đầu là `"aaaa"` ≠ `"area"`. Trọng tài `is_word_square` bắt được ngay.

In [29]:
def wrong_row_prefix(words):
    L = len(words[0]); pm = build_prefix_map(words)
    results, square = [], []
    def backtrack():
        r = len(square)
        if r == L:
            results.append(square[:]); return
        prefix = square[-1][:r] if square else ""    # SAI: đọc hàng, không phải cột
        for w in pm.get(prefix, []):
            square.append(w); backtrack(); square.pop()
    backtrack()
    return results

bad = wrong_row_prefix(words1)
print("Số kết quả:", len(bad), "| ví dụ:", bad[:3])
print("Có cái nào không hợp lệ?", any(not is_word_square(sq) for sq in bad))
show_square(bad[0])

Số kết quả: 7 | ví dụ: [['area', 'area', 'area', 'area'], ['lead', 'lead', 'lead', 'lead'], ['lead', 'lady', 'lady', 'lady']]
Có cái nào không hợp lệ? True
     c0 c1 c2 c3
r0   a  r  e  a     <- hàng 0 = 'area'
r1   a  r  e  a     <- hàng 1 = 'area'
r2   a  r  e  a     <- hàng 2 = 'area'
r3   a  r  e  a     <- hàng 3 = 'area'
Các cột đọc từ trên xuống: ['aaaa', 'rrrr', 'eeee', 'aaaa']
Hàng == cột ở mọi k? False


### Lỗi 4: Quên `pop()` và Lỗi 5: dừng ở `len(words)` thay vì `L`

**Lỗi 4 — quên bỏ chọn.** Không có `square.pop()`, từ đã thử ở nhánh trước **ở lại** trong `square`. Nhánh sau bắt đầu từ trạng thái bẩn: hàng 0 là `area` (ngõ cụt) nhưng không bị gỡ, nên `lead` bị đặt **chồng** lên thành hàng 1, rồi `wall` thành hàng 2, `lady` thành hàng 3 → ghi nhận hình vuông **sai** `[area, lead, wall, lady]`; sau đó `ball` thành hàng 5 và đọc `row[4]` của từ dài 4 → `IndexError`.

**Lỗi 5 — nhầm `N` với `L`.** Viết điều kiện dừng `len(square) == len(words)`. Với Example 2, `N = L = 4` nên **tình cờ** chạy đúng; với Example 1 (`N = 5`, `L = 4`), thuật toán cố đặt hàng thứ 5 và đọc `row[4]` → `IndexError`. Bài học: test cả ví dụ có `N ≠ L`.

Cả hai cell dưới bọc `try/except` để notebook chạy tiếp được và cho bạn thấy lỗi.

In [30]:
def wrong_no_pop(words, results):
    L = len(words[0]); pm = build_prefix_map(words)
    square = []
    def backtrack():
        r = len(square)
        if r == L:
            results.append(square[:]); return
        for w in pm.get("".join(row[r] for row in square), []):
            square.append(w); backtrack()            # SAI: thiếu square.pop()
    backtrack()
    return results

def wrong_stop_at_n(words, results):
    L = len(words[0]); pm = build_prefix_map(words)
    square = []
    def backtrack():
        r = len(square)
        if r == len(words):                          # SAI: phải là L
            results.append(square[:]); return
        for w in pm.get("".join(row[r] for row in square), []):
            square.append(w); backtrack(); square.pop()
    backtrack()
    return results

for fn, name, ws in [(wrong_no_pop, "words1", words1), (wrong_stop_at_n, "words2", words2),
                     (wrong_stop_at_n, "words1", words1)]:
    collected = []                                   # để xem được cả những gì đã ghi trước khi lỗi
    try:
        fn(ws, collected)
        ok = norm(collected) == norm(word_squares_hashmap(ws))
        print(f"{fn.__name__}({name}): {collected} -> {'đúng (tình cờ!)' if ok else 'SAI'}")
    except IndexError as e:
        print(f"{fn.__name__}({name}): IndexError: {e}")
        print(f"    đã kịp ghi nhận: {collected}",
              "| hợp lệ?", [is_word_square(sq) for sq in collected])

wrong_no_pop(words1): IndexError: string index out of range
    đã kịp ghi nhận: [['area', 'lead', 'wall', 'lady']] | hợp lệ? [False]
wrong_stop_at_n(words2): [['baba', 'abat', 'baba', 'atan'], ['baba', 'abat', 'baba', 'atal']] -> đúng (tình cờ!)
wrong_stop_at_n(words1): IndexError: string index out of range
    đã kịp ghi nhận: [] | hợp lệ? []


### Lỗi 6 (hiệu năng): quét lại toàn bộ `words` ở mỗi hàng

Cách 2 vẫn **đúng**, nhưng với `N = 1000` mỗi node phải gọi `startswith` 1000 lần. Đây là lý do bài là Hard: chạy ra đáp án không khó, chạy **kịp** mới là điểm cần học. Dấu hiệu nhận ra: trong vòng đệ quy có `for w in words: if w.startswith(...)` — hãy chuyển sang bảng tiền tố dựng sẵn một lần.

Cũng nhớ dùng `prefix_map.get(prefix, [])` thay vì `prefix_map[prefix]` khi bảng là `defaultdict`: đọc bằng `[]` sẽ âm thầm thêm hàng nghìn key rỗng (mục 8.1) — không sai kết quả nhưng phình bộ nhớ.

---

## 18. Mẹo nhận dạng pattern

- Đề yêu cầu **"trả về tất cả"** các cấu hình (hoán vị, tổ hợp, cách xếp, hình vuông...) → nghĩ ngay tới **backtracking**: chọn → đi tiếp → bỏ chọn.
- Khi chọn phần tử thứ `r`, các phần tử trước đã **cố định một phần** của nó (ở đây: `r` chữ đầu) → **tiền tố bắt buộc** → dựng **bảng tiền tố** (Hash Map tiền tố → danh sách, hoặc Trie) để lấy ứng viên tức thì.
- Có chữ "**cùng độ dài**", "**hàng = cột**", "**đối xứng**" → nghĩ tới chuyển vị `zip(*rows)` và điều kiện `a[i][j] == a[j][i]`.
- Thấy trong đệ quy có vòng `for` quét toàn bộ dữ liệu để lọc theo điều kiện "bắt đầu bằng" → tiền xử lý một lần thành `dict`/Trie.
- "Có thể dùng lại phần tử" → **không** có mảng `used`; "mỗi phần tử dùng một lần" → có `used` (xem Bài 5).

## 19. Biến thể & bài liên quan

| Bài | Liên hệ |
|---|---|
| #422 Valid Word Square (Easy, Premium) | Chỉ **kiểm tra** một dãy có phải word square không, các từ có thể khác độ dài (Bài 3) |
| #79 Word Search / #212 Word Search II | Backtracking + Trie trên lưới |
| #208 Implement Trie | Nền tảng Trie: `insert`, `search`, `startsWith` |
| #51 N-Queens / #37 Sudoku Solver | Backtracking đặt từng hàng, cắt nhánh sớm theo ràng buộc với các hàng trước |
| #46 Permutations / #78 Subsets | Khung backtracking cơ bản |
| #1065 Index Pairs of a String / #1268 Search Suggestions System | Tra cứu theo tiền tố bằng Trie / sắp xếp |

Biến thể tự nghĩ: mỗi từ chỉ dùng một lần trong một hình vuông (Bài 5); chỉ **đếm** số hình vuông; **forward checking** — cắt nhánh sớm hơn nữa bằng cách kiểm tra trước các hàng tương lai (Bài 6).

## 20. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 21) và **Đáp án** (mục 22).

### Bài 1 — Tính tay (không chạy code)

a) `["abc", "bde", "cef"]` có phải word square không? Đọc từng cột.

b) `["abc", "bdx", "cef"]` thì sao? Cặp ô đối xứng nào bị sai?

c) Cho `words = ["abc", "bde", "cef", "bdx"]`. Lập bảng trace như mục 10: với mỗi từ ở hàng 0, tiền tố của hàng 1 là gì, có ứng viên nào? Cuối cùng có bao nhiêu word square?

### Bài 2 — Đoán output

Không chạy, hãy đoán **chính xác** (kể cả thứ tự) kết quả của:

```python
word_squares_hashmap(["ab", "ba", "bb"])
```

Gợi ý suy nghĩ: `prefix_map[""]` giữ thứ tự xuất hiện trong `words`.

### Bài 3 — Điền TODO: #422 Valid Word Square

Cho `words` (các từ **có thể khác độ dài**), trả về `True` nếu hàng `k` và cột `k` đọc giống nhau với mọi `0 <= k < max(numRows, numColumns)`. Ví dụ của #422:

```text
["abcd","bnrt","crmy","dtye"] -> true
["abcd","bnrt","crm","dt"]    -> true
["ball","area","read","lady"] -> false
```

Lưu ý: khi các từ khác độ dài, ô `(j, i)` có thể **không tồn tại** — khi đó phải trả `False` thay vì để văng `IndexError`.

In [31]:
def valid_word_square(words: List[str]) -> bool:
    # TODO 1: duyệt i, row in enumerate(words)
    # TODO 2: duyệt j, ch in enumerate(row)
    # TODO 3: nếu hàng j không tồn tại, hoặc hàng j quá ngắn (không có cột i),
    #         hoặc words[j][i] != ch -> return False
    # TODO 4: đi hết mà không sai -> return True
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert valid_word_square(["abcd", "bnrt", "crmy", "dtye"]) is True
# assert valid_word_square(["abcd", "bnrt", "crm", "dt"]) is True
# assert valid_word_square(["ball", "area", "read", "lady"]) is False
# assert valid_word_square(["abc", "b"]) is False

### Bài 4 — Tự viết lại cách chính

Không nhìn mục 9, viết lại `exercise_word_squares(words)` gồm: dựng `prefix_map` bằng `defaultdict(list)` với tiền tố dài `0..L-1`, rồi backtracking với `square`. Nhớ ba chỗ hay sai: tiền tố lấy theo **cột**, lưu **bản sao**, có **`pop()`**.

In [32]:
def exercise_word_squares(words: List[str]) -> List[List[str]]:
    # TODO 1: L = len(words[0]); prefix_map = defaultdict(list); thêm w vào prefix_map[w[:k]] với k = 0..L-1
    # TODO 2: results, square = [], []
    # TODO 3: def backtrack(): r = len(square); nếu r == L -> lưu square[:]
    # TODO 4: prefix = cột r của các hàng đã đặt; duyệt prefix_map.get(prefix, []): chọn / đi tiếp / bỏ chọn
    # TODO 5: gọi backtrack() và trả results
    pass

# Tự làm xong rồi mới mở test:
# assert norm(exercise_word_squares(words1)) == norm(expected1)
# assert norm(exercise_word_squares(words2)) == norm(expected2)
# assert norm(exercise_word_squares(["a", "b"])) == [("a",), ("b",)]

### Bài 5 — Biến thể: mỗi từ dùng **tối đa một lần** trong một hình vuông

Viết `word_squares_distinct(words)` giống cách chính nhưng **cấm** dùng lại một từ trong cùng một hình vuông. Với Example 1 kết quả không đổi; với Example 2 kết quả là `[]` (vì cả hai đáp án đều dùng `baba` hai lần); với `["ab", "ba", "bb"]` (Bài 2 có 5 kết quả) chỉ còn 4 kết quả — mất đúng `["bb","bb"]`.

In [33]:
def word_squares_distinct(words: List[str]) -> List[List[str]]:
    # TODO 1: dựng prefix_map như cách chính
    # TODO 2: thêm một set `used`
    # TODO 3: khi duyệt ứng viên, bỏ qua w nếu w in used; chọn thì used.add(w), bỏ chọn thì used.remove(w)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert norm(word_squares_distinct(words1)) == norm(expected1)
# assert word_squares_distinct(words2) == []
# assert norm(word_squares_distinct(["ab", "ba", "bb"])) == norm([["ab", "ba"], ["ab", "bb"], ["ba", "ab"], ["bb", "ba"]])

### Bài 6 — Mini challenge: forward checking (cắt nhánh sớm hơn nữa)

Khi vừa đặt hàng `r`, ta đã biết thêm một chữ của **mọi hàng tương lai** `k > r`: tiền tố (đang dở) của hàng `k` là `"".join(row[k] for row in square)` (dài `r+1`). Nếu một tiền tố đó **không có** trong `prefix_map`, nhánh này chắc chắn chết — cắt ngay, không cần đi xuống.

Ví dụ: `square = ["lady"]` → hàng 2 sẽ phải bắt đầu bằng `"d"`, hàng 3 bằng `"y"`; không có từ nào bắt đầu bằng `"d"` → cắt luôn, khỏi thử `area` ở hàng 1.

Viết `word_squares_forward_check(words)` trả về **cặp** `(results, calls)` với `calls` = số lần gọi `backtrack()`. Lưu ý: `prefix_map` chỉ có tiền tố dài `<= L-1`, mà tiền tố của hàng tương lai dài `r+1 <= L-1` — vừa đủ. So với `count_backtrack_calls` (cho sẵn, đếm số lần gọi của cách chính), kết quả phải giống và số lần gọi phải **không nhiều hơn**.

In [34]:
def count_backtrack_calls(words):
    # Cho sẵn: số lần gọi backtrack() của cách chính (chạy được ngay)
    return sum(count_nodes(words))

print("Cách chính trên words1 gọi backtrack", count_backtrack_calls(words1), "lần")

def word_squares_forward_check(words: List[str]):
    # TODO 1: dựng prefix_map; results, square = [], []; calls = 0 (dùng list [0] hoặc nonlocal)
    # TODO 2: trong backtrack: calls += 1; r == L -> lưu
    # TODO 3: với mỗi ứng viên w: append(w); kiểm tra mọi k in range(r+1, L):
    #         "".join(row[k] for row in square) phải có trong prefix_map; hợp lệ hết mới backtrack()
    # TODO 4: pop(); cuối cùng trả (results, calls)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# res, calls = word_squares_forward_check(words1)
# assert norm(res) == norm(expected1) and calls <= count_backtrack_calls(words1)
# res, calls = word_squares_forward_check(big1000)
# assert norm(res) == norm(word_squares_hashmap(big1000)) and calls < count_backtrack_calls(big1000)

Cách chính trên words1 gọi backtrack 13 lần


## 21. Gợi ý theo mức

**Bài 1**
- Mức 1: viết 3 từ thành lưới 3x3, đọc từng cột từ trên xuống.
- Mức 2: ở b), so cặp `ô(1,2)` với `ô(2,1)`.
- Mức 3: ở c), chỉ `abc` ở hàng 0 cho tiền tố hàng 1 (`"b"`) có ứng viên; ba từ kia cho tiền tố `"d"`, `"e"`, `"d"`.

**Bài 2**
- Mức 1: `L = 2`, chỉ có 2 hàng; hàng 1 phải bắt đầu bằng chữ thứ 2 của hàng 0.
- Mức 2: `prefix_map["a"] = ["ab"]`, `prefix_map["b"] = ["ba", "bb"]`.
- Mức 3: có 5 kết quả, 2 cái đầu bắt đầu bằng `"ab"`.

**Bài 3**
- Mức 1: chỉ cần đi qua mọi ô `(i, j)` **có thật** trong hàng `i` và so với ô `(j, i)`.
- Mức 2: ô `(j, i)` tồn tại khi `j < len(words)` **và** `i < len(words[j])`.
- Mức 3: `if j >= len(words) or i >= len(words[j]) or words[j][i] != ch: return False`. Vì sao không cần kiểm tra chiều ngược lại (cột dài hơn hàng)? Vì ô thừa ở cột `i` là ô `(j, i)` có thật trong hàng `j` → sẽ bị kiểm tra khi duyệt hàng `j`.

**Bài 4**
- Mức 1: chép khuôn "chọn → đi tiếp → bỏ chọn" ở mục 6.
- Mức 2: tiền tố: `"".join(row[r] for row in square)`.
- Mức 3: `results.append(square[:])` và nhớ `square.pop()` sau `backtrack()`.

**Bài 5**
- Mức 1: giống tham số `used` trong #46 Permutations.
- Mức 2: `if w in used: continue`.
- Mức 3: `used.add(w)` ngay sau `append`, `used.remove(w)` ngay sau `pop`.

**Bài 6**
- Mức 1: kiểm tra **sau khi** `append(w)` để `square` đã có hàng mới.
- Mức 2: `all("".join(row[k] for row in square) in prefix_map for k in range(r + 1, L))`.
- Mức 3: kiểm tra `in prefix_map` (không đọc `prefix_map[...]` để khỏi tạo key rỗng trong `defaultdict`).

## 22. Đáp án tham khảo

### Đáp án Bài 1

```text
a)   a b c        cột 0: a b c = "abc" ✔
     b d e        cột 1: b d e = "bde" ✔
     c e f        cột 2: c e f = "cef" ✔     -> word square

b)   a b c        cột 1: b d e = "bde" nhưng hàng 1 = "bdx"
     b d x        sai cặp ô(1,2)='x' vs ô(2,1)='e'
     c e f                                    -> KHÔNG phải
```

c) Trace:

| Hàng 0 | Tiền tố hàng 1 | Ứng viên | Hàng 2: tiền tố | Ứng viên | Kết quả |
|---|---|---|---|---|---|
| `abc` | `"b"` | `bde, bdx` | `[abc,bde]` → `"ce"`; `[abc,bdx]` → `"cx"` | `cef`; không có | `[abc,bde,cef]` ✔ |
| `bde` | `"d"` | không có | — | — | ngõ cụt |
| `cef` | `"e"` | không có | — | — | ngõ cụt |
| `bdx` | `"d"` | không có | — | — | ngõ cụt |

→ Đúng **1** word square: `["abc", "bde", "cef"]`.

### Đáp án Bài 2

`[["ab","ba"], ["ab","bb"], ["ba","ab"], ["bb","ba"], ["bb","bb"]]` — hàng 0 lần lượt `ab, ba, bb` (thứ tự trong `words`); `ab` → tiền tố `"b"` → `ba, bb`; `ba` → `"a"` → `ab`; `bb` → `"b"` → `ba, bb`.

In [35]:
# Kiểm tra đáp án Bài 1 và Bài 2
print(is_word_square(["abc", "bde", "cef"]), is_word_square(["abc", "bdx", "cef"]))
ans1c = word_squares_hashmap(["abc", "bde", "cef", "bdx"])
print("Bài 1c:", ans1c)
assert ans1c == [["abc", "bde", "cef"]]

ans2 = word_squares_hashmap(["ab", "ba", "bb"])
print("Bài 2 :", ans2)
assert ans2 == [["ab", "ba"], ["ab", "bb"], ["ba", "ab"], ["bb", "ba"], ["bb", "bb"]]

True False
Bài 1c: [['abc', 'bde', 'cef']]
Bài 2 : [['ab', 'ba'], ['ab', 'bb'], ['ba', 'ab'], ['bb', 'ba'], ['bb', 'bb']]


In [36]:
# Đáp án Bài 3 (#422 Valid Word Square)
def answer_valid_word_square(words: List[str]) -> bool:
    for i, row in enumerate(words):
        for j, ch in enumerate(row):
            if j >= len(words) or i >= len(words[j]) or words[j][i] != ch:
                return False
    return True

assert answer_valid_word_square(["abcd", "bnrt", "crmy", "dtye"]) is True
assert answer_valid_word_square(["abcd", "bnrt", "crm", "dt"]) is True
assert answer_valid_word_square(["ball", "area", "read", "lady"]) is False
assert answer_valid_word_square(["abc", "b"]) is False
assert answer_valid_word_square(["a", "bc"]) is False     # hàng 1 dài hơn cột 1
print("Bài 3: PASS")

Bài 3: PASS


In [37]:
# Đáp án Bài 4
def answer_word_squares(words: List[str]) -> List[List[str]]:
    L = len(words[0])
    prefix_map = defaultdict(list)
    for w in words:
        for k in range(L):
            prefix_map[w[:k]].append(w)
    results, square = [], []

    def backtrack():
        r = len(square)
        if r == L:
            results.append(square[:])
            return
        prefix = "".join(row[r] for row in square)
        for w in prefix_map.get(prefix, []):
            square.append(w)
            backtrack()
            square.pop()

    backtrack()
    return results

assert norm(answer_word_squares(words1)) == norm(expected1)
assert norm(answer_word_squares(words2)) == norm(expected2)
assert norm(answer_word_squares(["a", "b"])) == [("a",), ("b",)]
print("Bài 4: PASS")

Bài 4: PASS


In [38]:
# Đáp án Bài 5
def answer_word_squares_distinct(words: List[str]) -> List[List[str]]:
    L = len(words[0])
    prefix_map = build_prefix_map(words)
    results, square, used = [], [], set()

    def backtrack():
        r = len(square)
        if r == L:
            results.append(square[:])
            return
        for w in prefix_map.get("".join(row[r] for row in square), []):
            if w in used:
                continue
            square.append(w); used.add(w)
            backtrack()
            square.pop(); used.remove(w)

    backtrack()
    return results

assert norm(answer_word_squares_distinct(words1)) == norm(expected1)
assert answer_word_squares_distinct(words2) == []
assert norm(answer_word_squares_distinct(["ab", "ba", "bb"])) == norm([["ab", "ba"], ["ab", "bb"], ["ba", "ab"], ["bb", "ba"]])
print("Bài 5: PASS")

Bài 5: PASS


In [39]:
# Đáp án Bài 6 (forward checking)
def answer_forward_check(words: List[str]):
    L = len(words[0])
    prefix_map = build_prefix_map(words)
    results, square = [], []
    calls = 0

    def backtrack():
        nonlocal calls
        calls += 1
        r = len(square)
        if r == L:
            results.append(square[:])
            return
        for w in prefix_map.get("".join(row[r] for row in square), []):
            square.append(w)
            if all("".join(row[k] for row in square) in prefix_map for k in range(r + 1, L)):
                backtrack()
            square.pop()

    backtrack()
    return results, calls

for name, ws in [("words1", words1), ("words2", words2), ("big1000", big1000)]:
    res, calls = answer_forward_check(ws)
    plain = count_backtrack_calls(ws)
    assert norm(res) == norm(word_squares_hashmap(ws)) and calls <= plain
    print(f"{name:8}: {len(res)} square | gọi backtrack: thường {plain:6}, forward checking {calls:6}")
print("Bài 6: PASS")

words1  : 2 square | gọi backtrack: thường     13, forward checking      9
words2  : 2 square | gọi backtrack: thường     15, forward checking      6


big1000 : 3268 square | gọi backtrack: thường  99740, forward checking  29864
Bài 6: PASS


## 23. Template nên nhớ

```python
from collections import defaultdict

class Solution:
    def wordSquares(self, words: List[str]) -> List[List[str]]:
        L = len(words[0])
        prefix_map = defaultdict(list)                 # 1. tiền tố -> danh sách từ
        for w in words:
            for k in range(L):                         #    tiền tố dài 0..L-1
                prefix_map[w[:k]].append(w)

        results, square = [], []
        def backtrack():
            r = len(square)
            if r == L:                                 # 2. đủ L hàng (không phải N!)
                results.append(square[:]); return      #    lưu BẢN SAO
            prefix = "".join(row[r] for row in square) # 3. cột r của các hàng trên
            for w in prefix_map.get(prefix, []):       # 4. chỉ ứng viên đúng tiền tố
                square.append(w)                       #    chọn
                backtrack()                            #    đi tiếp
                square.pop()                           #    bỏ chọn
        backtrack()
        return results
```

Câu thần chú: **"Hàng `r` bắt đầu bằng cột `r` — tra bảng tiền tố, chọn, đi tiếp, bỏ chọn; đủ `L` hàng thì chụp lại."**

## 24. Checklist trước khi submit

- [ ] Hiểu đúng: hàng `k` = cột `k` ⟺ `square[i][j] == square[j][i]`; số hàng = `L` (độ dài từ), **không phải** `N`.
- [ ] Từ được **dùng lại** → không có `used`, không dùng `permutations`.
- [ ] Tiền tố hàng `r` = `"".join(row[r] for row in square)` (đọc **dọc** cột `r`), hàng 0 có tiền tố `""`.
- [ ] `prefix_map` lưu tiền tố dài `0..L-1`; đọc bằng `.get(prefix, [])`.
- [ ] Ghi nhận `square[:]` (bản sao), mỗi `append` có đúng một `pop`.
- [ ] Thử tay: Example 1 (`N = 5 ≠ L = 4`) → 2 square; Example 2 (dùng `baba` 2 lần) → 2 square; `L = 1` → mỗi từ một square; so sánh bằng `sorted` vì thứ tự tùy ý.